# SuperStore

In [36]:
import pandas as pd
import polars as pl
import duckdb
import numpy as np

from modules.const import DUCKDB_FILE_PATH

with duckdb.connect(DUCKDB_FILE_PATH) as conn:
    facts     = conn.sql("SELECT * FROM FactSales").pl().lazy()
    customers = conn.sql("SELECT * FROM DimCustomers").pl().lazy()
    products  = conn.sql("SELECT * FROM DimProducts").pl().lazy()
    geography = conn.sql("SELECT * FROM DimGeography").pl().lazy()
    conn.sql("DESCRIBE FactSales").show()

print(*[f"{sc[0]:<12}- {sc[1]}" for sc in facts.collect_schema().items()], sep='\n')

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ Row_ID      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ OrderID     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ OrderDate   │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ShipDate    │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ ShipMode    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ProductKey  │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ CustomerKey │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ GeoKey      │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ Sales       │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

Row_ID      - Int64

## 1. YoY Revenue growth per Category  

|   Year | Category        |   Revenue |   Prev_Revenue |   yoy_pct |
|-------:|:----------------|----------:|---------------:|----------:|
|   2015 | Furniture       |    156181 |            nan |    nan    |
|   2016 | Furniture       |    162827 |         156181 |      4.26 |
|   2017 | Furniture       |    194785 |         162827 |     19.63 |
|   2018 | Furniture       |    211270 |         194785 |      8.46 |
|   2015 | Office Supplies |    148988 |            nan |    nan    |
|   2016 | Office Supplies |    131935 |         148988 |    -11.45 |
|   2017 | Office Supplies |    180994 |         131935 |     37.18 |
|   2018 | Office Supplies |    237116 |         180994 |     31.01 |
|   2015 | Technology      |    170773 |            nan |    nan    |
|   2016 | Technology      |    155366 |         170773 |     -9.02 |
|   2017 | Technology      |    219354 |         155366 |     41.19 |
|   2018 | Technology      |    260477 |         219354 |     18.75 |

In [87]:
(
    facts
    .select(['OrderDate', 'ProductKey', 'Sales'])
    .with_columns(Year=pl.col('OrderDate').dt.year())
    .join(
        products.cast({'ProductKey': pl.Float64}), on='ProductKey', how='inner'
    )
    .group_by(['Year', 'Category'])
    .agg(Revenue=pl.col('Sales').sum())
    .with_columns(
        Prev_Revenue=pl.col('Revenue')
            .shift(1)
            .over(partition_by='Category', order_by='Year')
            .round(2)
    )
    .with_columns(
        yoy_pct=((pl.col('Revenue') / pl.col('Prev_Revenue')) * 100).round(2)
    )
    
    .sort(['Category', 'Year'])
).collect()

Year,Category,Revenue,Prev_Revenue,yoy_pct
i32,str,f64,f64,f64
2015,"""Furniture""",156181.0671,null,null
2016,"""Furniture""",162826.9684,156181.07,104.26
2017,"""Furniture""",194785.3,162826.97,119.63
2018,"""Furniture""",211269.7732,194785.3,108.46
2015,"""Office Supplies""",148987.972,null,null
…,…,…,…,…
2018,"""Office Supplies""",237115.691,180993.53,131.01
2015,"""Technology""",170773.233,null,null
2016,"""Technology""",155365.901,170773.23,90.98


## 2. Customer cohort retention  

|   Cohort_Year |   Order_Year |   Periods_Since_First |   Customers |   Revenue |   Avg_Revenue_per_Customer |
|--------------:|-------------:|----------------------:|------------:|----------:|---------------------------:|
|          2015 |         2015 |                     0 |         589 |  479856   |                     814.7  |
|          2015 |         2016 |                     1 |         426 |  347204   |                     815.03 |
|          2015 |         2017 |                     2 |         476 |  445145   |                     935.18 |
|          2015 |         2018 |                     3 |         510 |  522227   |                    1023.98 |
|          2016 |         2016 |                     0 |         141 |  112232   |                     795.97 |
|          2016 |         2017 |                     1 |         107 |  100663   |                     940.77 |
|          2016 |         2018 |                     2 |         124 |  131200   |                    1058.06 |
|          2017 |         2017 |                     0 |          52 |   54385.2 |                    1045.87 |
|          2017 |         2018 |                     1 |          45 |   61112.9 |                    1358.07 |
|          2018 |         2018 |                     0 |          11 |    7511.8 |                     682.89 |

In [89]:
(
    facts
    .with_columns(
        Order_Year=pl.col('OrderDate').dt.year(),
        Cohort_Year=pl.col('OrderDate').min().over('CustomerKey').dt.year(),
    )
    .with_columns(Periods_Since_First=pl.col('Order_Year') - pl.col('Cohort_Year'))
    .group_by(['Cohort_Year', 'Order_Year', 'Periods_Since_First'])
    .agg(
        Customers=pl.col('CustomerKey').n_unique(),
        Revenue=pl.col('Sales').sum().round(2)
    )
    .with_columns(Avg_Revenue_per_Customer=(pl.col('Revenue') / pl.col('Customers')).round(2))
).collect()

Cohort_Year,Order_Year,Periods_Since_First,Customers,Revenue,Avg_Revenue_per_Customer
i32,i32,i32,u32,f64,f64
2015,2015,0,589,479856.21,814.7
2016,2016,0,141,112231.66,795.97
2017,2017,0,52,54385.17,1045.87
2018,2018,0,11,7511.8,682.89
2017,2018,1,45,61112.93,1358.07
2015,2018,3,510,522227.45,1023.98
2015,2016,1,426,347204.34,815.03
2016,2018,2,124,131199.83,1058.06
2016,2017,1,107,100662.82,940.77


## 3. ABC analysis sui prodotti  

| ProductID       | ProductName                                                                 | Category        | SubCategory   |   Revenue |   revenue_pct |   cumulative_pct | class_   |
|:----------------|:----------------------------------------------------------------------------|:----------------|:--------------|----------:|--------------:|-----------------:|:---------|
| TEC-CO-10004722 | Canon imageCLASS 2200 Advanced Copier                                       | Technology      | Copiers       |   61599.8 |      2.7238   |          2.7238  | A        |
| OFF-BI-10003527 | Fellowes PB500 Electric Punch Plastic Comb Binding Machine with Manual Bind | Office Supplies | Binders       |   27453.4 |      1.21393  |          3.93773 | A        |
| TEC-MA-10002412 | Cisco TelePresence System EX90 Videoconferencing Unit                       | Technology      | Machines      |   22638.5 |      1.00102  |          4.93875 | A        |
| FUR-CH-10002024 | HON 5400 Series Task Chairs for Big and Tall                                | Furniture       | Chairs        |   21870.6 |      0.967067 |          5.90582 | A        |
| OFF-BI-10001359 | GBC DocuBind TL300 Electric Binding System                                  | Office Supplies | Binders       |   19823.5 |      0.876549 |          6.78237 | A        |

In [ ]:
(
    products
    .select(['ProductKey', 'ProductID', 'ProductName', 'Category', 'SubCategory'])
    .cast({'ProductKey': pl.Float64})
    .join(facts.select(['ProductKey', 'Sales']), on='ProductKey', how='inner')
    .group_by(['ProductID', 'ProductName', 'Category', 'SubCategory'])
    .agg(Revenue=pl.col('Sales').sum())
    .with_columns(
        revenue_pct=pl.col('Revenue') / facts.select('Sales').sum().collect().item() * 100
    )
    .sort('revenue_pct', descending=True)
    .with_columns(
        cumulative_pct=pl.col('revenue_pct').cum_sum().round(6),
    )
    .with_columns(
        class_=pl.col('cumulative_pct').cut(
            breaks=[80, 95],
            labels=['A', 'B', 'C']
        ),
        Revenue=pl.col('Revenue').round(2),
        revenue_pct=pl.col('revenue_pct').round(6),
        cumulative_pct=pl.col('cumulative_pct').round(6)
    )
).collect()

ProductID,ProductName,Category,SubCategory,Revenue,revenue_pct,cumulative_pct,class_
str,str,str,str,f64,f64,f64,cat
"""TEC-CO-10004722""","""Canon imageCLASS 2200 Advanced…","""Technology""","""Copiers""",61599.82,2.723804,2.723804,"""A"""
"""OFF-BI-10003527""","""Fellowes PB500 Electric Punch …","""Office Supplies""","""Binders""",27453.38,1.213926,3.93773,"""A"""
"""TEC-MA-10002412""","""Cisco TelePresence System EX90…","""Technology""","""Machines""",22638.48,1.001022,4.938752,"""A"""
"""FUR-CH-10002024""","""HON 5400 Series Task Chairs fo…","""Furniture""","""Chairs""",21870.58,0.967067,5.905819,"""A"""
"""OFF-BI-10001359""","""GBC DocuBind TL300 Electric Bi…","""Office Supplies""","""Binders""",19823.48,0.876549,6.782368,"""A"""
…,…,…,…,…,…,…,…
"""OFF-SU-10003936""","""Acme Serrated Blade Letter Ope…","""Office Supplies""","""Supplies""",7.63,0.000337,98.607473,"""C"""
"""OFF-EN-10001535""","""Grip Seal Envelopes""","""Office Supplies""","""Envelopes""",7.07,0.000313,98.607785,"""C"""
"""OFF-PA-10000048""","""Xerox 20""","""Office Supplies""","""Paper""",6.48,0.000287,98.608072,"""C"""


## 4. Shipping performance  

| ShipMode       |   Avg_Delay_Days |   Min_Delay_Days |   Max_Delay_Days |   Expected_Min |   Expected_Max |   Anomalies |   Anomaly_pct |
|:---------------|-----------------:|-----------------:|-----------------:|---------------:|---------------:|------------:|--------------:|
| First Class    |             2.18 |                1 |                4 |              1 |              3 |           1 |         0.067 |
| Second Class   |             3.25 |                1 |                5 |              2 |              5 |           1 |         0.053 |
| Standard Class |             5.01 |                3 |                7 |              4 |              7 |           1 |         0.017 |
| Same Day       |             0.04 |                0 |                1 |              0 |              1 |           0 |         0     |

In [211]:
(
    facts
    .cast({
        'ShipMode': pl.Enum(
            ['Same Day', 'Standard Class', 'First Class', 'Second Class']
        )
    })
    .with_columns(
        delta_days=(pl.col('ShipDate') - pl.col('OrderDate')).dt.total_days(),

        Expected_Min=pl.col('ShipMode').replace_strict({
            'Same Day': 0,
            'Standard Class': 4,
            'First Class': 1,
            'Second Class': 2
        }, default=None)
        .cast(pl.Int8),

        Expected_Max=pl.col('ShipMode').replace_strict({
            'Same Day': 1,
            'Standard Class': 7,
            'First Class': 3,
            'Second Class': 5
        }, default=None)
        .cast(pl.Int8)
    )
    .with_columns(
        isAnomaly=(
            (pl.col('delta_days') < pl.col('Expected_Min')) |
            (pl.col('delta_days') > pl.col('Expected_Max'))
        )
    )
    .group_by('ShipMode')
    .agg(
        Avg_Delay_Days=pl.col('delta_days').mean().round(2),
        Min_Delay_Days=pl.col('delta_days').min(),
        Max_Delay_Days=pl.col('delta_days').max(),
        Expected_Min=pl.col('Expected_Min').first(),
        Expected_Max=pl.col('Expected_Max').first(),
        Anomalies=pl.col('isAnomaly').sum(),
        Anomaly_pct=(pl.col('isAnomaly').mean() * 100).round(3)
    )
    .sort('Anomaly_pct', descending=True)
).collect()

ShipMode,Avg_Delay_Days,Min_Delay_Days,Max_Delay_Days,Expected_Min,Expected_Max,Anomalies,Anomaly_pct
enum,f64,i64,i64,i8,i8,u32,f64
"""First Class""",2.18,1,4,1,3,1,0.067
"""Second Class""",3.25,1,5,2,5,1,0.053
"""Standard Class""",5.01,3,7,4,7,1,0.017
"""Same Day""",0.04,0,1,0,1,0,0.0


## 5. Geographic market penetration  

| Region   | State     |   Orders |   Revenue |   Avg_Order_Value |   National_Revenue_Pct |   Revenue_Rank | Segment   |
|:---------|:----------|---------:|----------:|------------------:|-----------------------:|---------------:|:----------|
| Central  | Texas     |      480 |  168573   |            173.25 |                   7.45 |              1 | Other     |
| Central  | Illinois  |      270 |   79236.5 |            164.05 |                   3.5  |              2 | Other     |
| Central  | Michigan  |      116 |   76136.1 |            300.93 |                   3.37 |              3 | Other     |
| Central  | Indiana   |       70 |   48718.4 |            360.88 |                   2.15 |              4 | Other     |
| Central  | Wisconsin |       52 |   31173.4 |            296.89 |                   1.38 |              5 | Other     |

In [235]:
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(
        facts
        .join(geography, on='GeoKey', how='left')
        .group_by(['Region', 'State'])
        .agg(
            Orders=pl.col('OrderID').n_unique(),
            Revenue=pl.col('Sales').sum().round(2),
            Avg_Order_Value=pl.col('Sales').mean().round(2)
        )
        .with_columns(
            National_Revenue_pct=(pl.col('Revenue') / pl.col('Revenue').sum() * 100).round(2),
            Revenue_Rank=pl.col('Revenue').rank('min', descending=True).over('Region'),
            Segment=pl
                .when((pl.col('Orders') >= pl.col('Orders').quantile(0.5)) & (pl.col('Revenue') <= pl.col('Revenue').quantile(0.5)))
                .then(pl.lit('High Volume Low Ticket'))
                .when((pl.col('Orders') <= pl.col('Orders').quantile(0.5)) & (pl.col('Revenue') >= pl.col('Revenue').quantile(0.5)))
                .then(pl.lit('Low Volume High Ticket'))
                .otherwise(pl.lit('Other'))
        )
        .sort(['Region', 'Revenue_Rank'])
        .collect()
    )

Region,State,Orders,Revenue,Avg_Order_Value,National_Revenue_pct,Revenue_Rank,Segment
str,str,u32,f64,f64,f64,u32,str
"""Central""","""Texas""",480,168572.53,173.25,7.45,1,"""Other"""
"""Central""","""Illinois""",270,79236.52,164.05,3.5,2,"""Other"""
"""Central""","""Michigan""",116,76136.07,300.93,3.37,3,"""Other"""
"""Central""","""Indiana""",70,48718.4,360.88,2.15,4,"""Other"""
"""Central""","""Wisconsin""",52,31173.43,296.89,1.38,5,"""Other"""
"""Central""","""Minnesota""",44,29863.15,335.54,1.32,6,"""Other"""
"""Central""","""Missouri""",30,22205.15,336.44,0.98,7,"""Low Volume High Ticket"""
"""Central""","""Oklahoma""",34,19683.39,298.23,0.87,8,"""Other"""
"""Central""","""Nebraska""",23,7464.93,196.45,0.33,9,"""Other"""
